In [ ]:
# ==========================================================
# セル1：ライブラリと共通モジュール fft_common の読み込み・解析設定
# ==========================================================
# 【対象】Futaba形式（1行目が "Time:" で始まるCSV）
#   もう一方の形式は 06_NR500複数まとめファイル用_圧力経時変化フーリエ変換.ipynb を使ってください。
#
# ★加工対象CSVには一切書き込まない（読み取り専用）★
# ==========================================================
import os
import sys
import time
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog

# --- 共通モジュール fft_common.py を探して読み込む ---
# ノートブックの実行時カレントディレクトリが一定しないため、
# 想定される場所を順に探す。見つからなければ理由を明示して止める。
_here = os.getcwd()
_cands = [
    _here,
    os.path.join(_here, "コードフォルダ"),
    os.path.dirname(_here),
    os.path.join(os.path.dirname(_here), "コードフォルダ"),
]
for _c in _cands:
    if os.path.isfile(os.path.join(_c, "fft_common.py")):
        if _c not in sys.path:
            sys.path.insert(0, _c)
        break
else:
    raise FileNotFoundError(
        "fft_common.py が見つかりません。\n探した場所:\n  "
        + "\n  ".join(_cands)
        + "\nコードフォルダと同じ場所（またはその親フォルダ）で実行してください。"
    )

import fft_common
from fft_common import (
    VOLT_TO_MPA, amp_spectrum, detect_format, folder_tag, list_csv_by_format,
    metrics, read_any, setup_japanese_font,
)

setup_japanese_font(plt)
warnings.filterwarnings("ignore", category=RuntimeWarning,
                        message="All-NaN slice encountered")

# ==========================================================
# ⚙️ 解析設定（ここを変えると解析方法が変わります）
# ==========================================================
TARGET_FORMAT = "futaba"
CHANNELS = ["CH03", "CH04"]

# --- ゼロ点合わせ（先頭 N 行の平均を全行から引く）---
ZERO_ADJUST = False      # Futabaは元からMPaで基準線も0近傍のため既定OFF
ZERO_ROWS = 1000                    # ゼロ点合わせに使う先頭行数

# --- スペクトルの求め方 ---
USE_EVENT_WINDOW = True   # True : 圧力が立ち上がっているイベント区間だけを解析（推奨）
                          # False: 記録全体を解析（＝従来の動作。無信号区間のノイズが混入）
DETREND = True            # FFT前に解析区間の平均を引く（False が従来の動作）
WINDOW = "hann"           # "hann"（漏れが少ない）/ "none"（＝従来の矩形窓）
# ==========================================================
# ※グラフの見た目（周波数軸の範囲・対数軸など）は セル4 にあります

print("✅ 準備完了")
print(f"   共通モジュール : {fft_common.__file__}")
print(f"   対象形式       : {TARGET_FORMAT}   チャンネル: {', '.join(CHANNELS)}")
if ZERO_ADJUST:
    print(f"   ゼロ点合わせ   : ON（先頭 {ZERO_ROWS} 行の平均を全行から引く）")
else:
    print("   ゼロ点合わせ   : OFF")
print(f"   解析区間       : {'イベント区間のみ' if USE_EVENT_WINDOW else '記録全体（従来動作）'}")
print(f"   窓関数         : {WINDOW} / 直流除去: {DETREND}")
print("👉 次のセルを実行してください。")

In [ ]:
# ==========================================================
# セル2：メインフォルダの選択
# ==========================================================
root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)
main_dir = filedialog.askdirectory(title="解析対象のメインフォルダを選択してください")
root.destroy()

if not main_dir:
    print("⚠️ フォルダ選択がキャンセルされました。次のセルには進まず、やり直してください。")
else:
    print(f"✅ 選択されたメインフォルダ:\n{main_dir}")

In [ ]:
# ==========================================================
# セル3：対象ファイルの列挙と形式の絞り込み
# ==========================================================
if not main_dir:
    raise ValueError("メインフォルダが選択されていません。セル2を再実行してください。")

print("🔍 CSVを走査中...（1行目を見て形式を判定します）")
buckets = list_csv_by_format(main_dir)
target_files = buckets[TARGET_FORMAT]
others = sum(([p] for k, v in buckets.items() if k != TARGET_FORMAT for p in v), [])
n_all = sum(len(v) for v in buckets.values())

print(f"\n見つかったCSVファイル: 合計 {n_all} 件")
print(f"   ├ Futaba形式 : {len(target_files):5d} 件 ← 処理します")
for k, label in (("futaba", "Futaba形式"), ("nr500", "NR-500形式"), ("other", "対象外")):
    if k != TARGET_FORMAT and buckets[k]:
        print(f"   ├ {label:12s} : {len(buckets[k]):5d} 件 ← 除外します")

if target_files:
    print("\n【処理対象ファイルの例】")
    for f in target_files[:3]:
        print(" -", f)
    if len(target_files) > 3:
        print(f"   ... (他 {len(target_files) - 3} 件)")
else:
    print("\n⚠️ Futaba形式のCSVが1件も見つかりませんでした。")

# 除外したファイルも必ず見せる（無言でスキップしない）
if others:
    print("\n【除外したファイルの例】")
    for f in others[:5]:
        print(f" - {os.path.basename(f)}  ({detect_format(f) or '対象外'})")
    if len(others) > 5:
        print(f"   ... (他 {len(others) - 5} 件)")
    print("   → もう一方の形式は 06_NR500複数まとめファイル用_圧力経時変化フーリエ変換.ipynb で処理できます。")

if target_files:
    print(f"\n✅ {len(target_files)} 件を処理します。次のセルへ進んでください。")

In [ ]:
# ==========================================================
# セル4：グラフ設定とファイルのグループ化（準備）
# ==========================================================
if not target_files:
    raise ValueError("処理対象ファイルがありません。セル3を確認してください。")

# ==========================================
# ⚙️ グラフの設定
# ※ セル1にも解析設定（イベント区間・窓関数・ゼロ点合わせ）があります
# ==========================================
LOG_Y = True                          # Y軸を対数にする（False が従来のリニア軸）
X_MIN, X_MAX, X_STEP = 0, 150, 10     # 周波数軸 [Hz]
Y_MIN, Y_MAX = 1e-4, None             # 縦軸（LOG_Y=True のときの下限／上限。None で自動）
SHOW_PERCENTILE_BAND = True           # 中央値と95パーセンタイルの帯を重ねる
PERCENTILE_HI = 95

# キャッシュする周波数の上限。X_MAX をこれより広げたい場合は
# ここを増やして「セル5」から実行し直してください。
CACHE_MAX_HZ = 500
CACHE_POINTS = 2000
# ==========================================

output_dir_path = os.path.join(os.getcwd(), "fft_results_per_channel")
os.makedirs(output_dir_path, exist_ok=True)

# 末端フォルダ（＝温度・板厚などの条件）ごとにグループ化
folder_to_files = defaultdict(list)
for path in target_files:
    folder_to_files[os.path.dirname(path)].append(path)

# 全ファイル共通の周波数グリッド（解析区間の長さが違ってもパーセンタイルを取れる）
GRID_EDGES = np.linspace(0.0, CACHE_MAX_HZ, CACHE_POINTS + 1)
GRID_F = 0.5 * (GRID_EDGES[:-1] + GRID_EDGES[1:])

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

print("✅ 準備完了")
print(f"📁 出力先        : {output_dir_path}")
print(f"🎯 チャンネル    : {', '.join(CHANNELS)}")
print(f"📊 X軸           : {X_MIN}〜{X_MAX} Hz（目盛り {X_STEP} Hz）")
print(f"📊 Y軸           : {'対数' if LOG_Y else 'リニア'}"
      f"（下限 {Y_MIN} / 上限 {'自動' if Y_MAX is None else Y_MAX}）")
print(f"📊 パーセンタイル帯: {'表示する' if SHOW_PERCENTILE_BAND else '表示しない'}")
print(f"💾 キャッシュ上限 : {CACHE_MAX_HZ} Hz（{CACHE_POINTS} 点）")
print(f"📦 条件グループ  : {len(folder_to_files)} 件 / ファイル {len(target_files)} 件")
print(f"\n⏱️ セル5の目安  : 約 {len(target_files) * 0.15 / 60:.1f} 分")
print("👉 次の「セル5」を実行してください。")

In [ ]:
# ==========================================================
# セル5：全ファイルのスペクトルと指標を一括計算（重い処理はここだけ）
# ==========================================================
# ファイルは1回しか読みません。共通グリッド上のスペクトルをメモリに保持するので、
# セル6・セル7の描画は一瞬で終わり、軸設定を変えて描き直すのも即座です。
# （X_MAX を CACHE_MAX_HZ より広げたい場合だけ、セル4→セル5からやり直し）
# ==========================================================
from fft_common import bin_to_grid

print("🚀 一括計算を開始します...")
t0 = time.time()

spec_cache = {}     # (パス, ch) -> 共通グリッド上の振幅スペクトル [MPa]
records = []        # 1ファイル×1チャンネル につき1行の指標
errors = []

for i, path in enumerate(target_files, 1):
    try:
        df, dt, info = read_any(path, channels=CHANNELS,
                                zero=ZERO_ADJUST, zero_rows=ZERO_ROWS)
    except Exception as e:
        errors.append((path, f"{type(e).__name__}: {e}"))
        continue

    for ch in CHANNELS:
        try:
            y = df[ch].to_numpy(dtype=np.float64)
            m, _spec, (i0, i1) = metrics(
                y, dt, use_event=USE_EVENT_WINDOW, detrend=DETREND, window=WINDOW,
                zero_rows=ZERO_ROWS if info["zero_applied"] else None)
            f, a = amp_spectrum(y[i0:i1 + 1], dt, detrend=DETREND, window=WINDOW)
            spec_cache[(path, ch)] = bin_to_grid(f, a, GRID_EDGES)

            rec = {"file": os.path.basename(path), "path": path,
                   "group": folder_tag(os.path.dirname(path), main_dir),
                   "format": info["format"], "channel": ch,
                   "zero_offset_applied_MPa": info["zero_offset"].get(ch, 0.0)}
            rec.update(m)
            records.append(rec)
        except Exception as e:
            errors.append((path, f"[{ch}] {type(e).__name__}: {e}"))

    if i % 25 == 0 or i == len(target_files):
        el = time.time() - t0
        print(f"  {i}/{len(target_files)} ファイル（経過 {el:.0f}秒 / "
              f"残り約 {el / i * (len(target_files) - i):.0f}秒）")

met = pd.DataFrame(records)

print("\n" + "-" * 52)
print(f"🏁 一括計算 完了！  {len(met)} 行  所要 {time.time() - t0:.0f} 秒")

# ---- 品質フラグの集計（無言でスキップしない）----
if len(met):
    print("\n【品質フラグの内訳】")
    for flag, cnt in met["flags"].value_counts().items():
        print(f"  {'✅' if flag == 'ok' else '⚠️'} {flag:32s} {cnt:5d} 件")
    bad = met[met["flags"] != "ok"]
    if len(bad):
        print(f"\n  ⚠️ 要確認 {len(bad)} 件（先頭10件。全件はセル8のCSVを参照）:")
        for _, r in bad.head(10).iterrows():
            print(f"     {r['file']} [{r['channel']}] → {r['flags']}")

if errors:
    print(f"\n❌ 失敗 {len(errors)} 件:")
    for p, msg in errors[:10]:
        print(f"   {os.path.basename(p)}: {msg}")

print(f"\n💾 キャッシュ: {len(spec_cache)} 系列 × {len(GRID_F)} 点 "
      f"（約 {len(spec_cache) * len(GRID_F) * 8 / 1e6:.1f} MB）")
print("👉 次の「セル6」を実行してください。")

In [ ]:
# ==========================================================
# セル6：チャンネルごとの重ね描きグラフ（個別版）
# ==========================================================
# セル5のキャッシュから描くだけなので一瞬で終わります。
# 軸設定を変えたいときは「セル4を書き換えてセル4→セル6」でOK（セル5の再実行は不要）。
# ==========================================================
sel = (GRID_F >= X_MIN) & (GRID_F <= X_MAX)


def _style_axes(ax, title):
    ax.set_xlabel("Frequency [Hz]")
    ax.set_ylabel("Amplitude [MPa]")
    ax.set_title(title)
    ax.set_xlim(X_MIN, X_MAX)
    ax.set_xticks(np.arange(X_MIN, X_MAX + X_STEP, X_STEP))
    if LOG_Y:
        ax.set_yscale("log")
        ax.set_ylim(Y_MIN, Y_MAX)
    elif Y_MAX is not None:
        ax.set_ylim(0, Y_MAX)
    ax.grid(True, which="both", linestyle="--", alpha=0.5)


def _line_alpha(n):
    """重ねる本数に応じて線の濃さを変える（少数なら濃く、多数なら薄く）。"""
    return float(np.clip(4.0 / max(n, 1), 0.12, 0.85))


print("🚀 【個別版】チャンネルごとの重ね描きグラフを作成します...")
n_saved = 0

for dirpath, files_in_dir in folder_to_files.items():
    tag = folder_tag(dirpath, main_dir)

    for ch in CHANNELS:
        curves = [spec_cache[(p, ch)] for p in files_in_dir if (p, ch) in spec_cache]
        if not curves:
            continue

        fig, ax = plt.subplots(figsize=(10, 5))
        al = _line_alpha(len(curves))
        for c in curves:
            ax.plot(GRID_F[sel], c[sel], color="gray", alpha=al, linewidth=0.8)

        if SHOW_PERCENTILE_BAND and len(curves) > 1:
            # 全ファイルを重ねただけでは包絡しか読めないので、
            # 中央値と上側パーセンタイルを重ねて定量的に読めるようにする
            arr = np.vstack(curves)
            med = np.nanpercentile(arr, 50, axis=0)
            hi = np.nanpercentile(arr, PERCENTILE_HI, axis=0)
            ax.plot(GRID_F[sel], med[sel], color="tab:blue", linewidth=2.0,
                    label=f"中央値 (n={len(curves)})")
            ax.plot(GRID_F[sel], hi[sel], color="tab:red", linewidth=1.4,
                    linestyle="--", label=f"{PERCENTILE_HI}パーセンタイル")
            ax.legend(loc="upper right", fontsize=9)

        _style_axes(ax, f"FFT Spectrum － {tag} [{ch}]（{len(curves)} ファイル重ね描き）")
        fig.tight_layout()
        # 相対パスをタグに使うので 0.5mm/190℃ と 1.0mm/190℃ が衝突しない
        fig.savefig(os.path.join(
            output_dir_path, f"{tag}_{ch}_combined_fft.png"), dpi=150)
        plt.close(fig)
        n_saved += 1
        print(f"  --> ✅ 保存: {tag}_{ch}_combined_fft.png")

print("-" * 52)
print(f"🏁 個別グラフ完了！（生成 {n_saved} 枚）")
print("👉 次の「セル7」を実行してください。")

In [ ]:
# ==========================================================
# セル7：全チャンネルをまとめた統合グラフ（統合版）
# ==========================================================
print("🚀 【統合版】全チャンネルのグラフを作成します...")
n_saved = 0

for dirpath, files_in_dir in folder_to_files.items():
    tag = folder_tag(dirpath, main_dir)
    parent_dir_path = os.path.dirname(dirpath)

    fig, ax = plt.subplots(figsize=(12, 6))
    drawn_files = set()
    labelled = set()

    for i, ch in enumerate(CHANNELS):
        curves = [spec_cache[(p, ch)] for p in files_in_dir if (p, ch) in spec_cache]
        if not curves:
            continue
        color = colors[i % len(colors)]
        al = _line_alpha(len(curves))
        for p in files_in_dir:
            if (p, ch) not in spec_cache:
                continue
            c = spec_cache[(p, ch)]
            label = ch if ch not in labelled else ""
            ax.plot(GRID_F[sel], c[sel], color=color, alpha=al,
                    linewidth=0.8, label=label)
            labelled.add(ch)
            drawn_files.add(p)

        if SHOW_PERCENTILE_BAND and len(curves) > 1:
            med = np.nanpercentile(np.vstack(curves), 50, axis=0)
            ax.plot(GRID_F[sel], med[sel], color=color, linewidth=2.2,
                    label=f"{ch} 中央値 (n={len(curves)})")

    if not drawn_files:
        plt.close(fig)
        continue

    ax.set_xlabel("Frequency [Hz]")
    ax.set_ylabel("Amplitude [MPa]")
    fig.suptitle(f"Path: {parent_dir_path}", fontsize=10, color="dimgray", y=0.99)
    ax.set_title(f"FFT Spectrum － {tag} [ALL CHANNELS]"
                 f"（{len(drawn_files)} ファイル重ね描き）")
    ax.set_xlim(X_MIN, X_MAX)
    ax.set_xticks(np.arange(X_MIN, X_MAX + X_STEP, X_STEP))
    if LOG_Y:
        ax.set_yscale("log")
        ax.set_ylim(Y_MIN, Y_MAX)
    elif Y_MAX is not None:
        ax.set_ylim(0, Y_MAX)
    ax.grid(True, which="both", linestyle="--", alpha=0.5)
    ax.legend(loc="upper right", fontsize=9)
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    fig.savefig(os.path.join(
        output_dir_path, f"{tag}_ALL_combined_fft.png"), dpi=150)
    n_saved += 1
    print(f"  --> 🌟 保存: {tag}_ALL_combined_fft.png")
    plt.show()

print("-" * 52)
print(f"🏁 統合グラフ完了！（生成 {n_saved} 枚）")
print(f"📁 出力先: {output_dir_path}")
print("👉 次の「セル8」を実行してください。")

In [ ]:
# ==========================================================
# セル8：指標のCSV出力と、条件ごとのまとめ
# ==========================================================
if "met" not in globals() or len(met) == 0:
    raise ValueError("指標がありません。セル5を実行してください。")

csv_path = os.path.join(output_dir_path, "metrics_per_file.csv")
met.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"💾 指標を保存しました（Excelでそのまま開けます）\n   {csv_path}")
print(f"   {len(met)} 行（1ファイル×1チャンネル）")

# ---- 条件グループごとのまとめ ----
print("\n【条件ごとのまとめ（中央値）】")
head = (f"{'条件':<22} {'n':>4} {'ピーク':>9} {'立上り':>9} "
        f"{'f99':>8} {'f99.9':>8} {'ノイズRMS':>10}")
print(head)
print("-" * len(head))
for g, sub in met.groupby("group"):
    print(f"{g:<22} {len(sub):4d} "
          f"{sub['peak_MPa'].median():8.2f}  "
          f"{sub['rise_time_ms'].median():7.1f}ms "
          f"{sub['f99_Hz'].median():7.2f} {sub['f999_Hz'].median():7.2f} "
          f"{sub['noise_rms_MPa'].median():9.4f}")

print("\n【全体（カットオフ決定の目安）】")
for col, name in (("f99_Hz", "f99"), ("f999_Hz", "f99.9"),
                  ("required_bw_Hz", "立上りからの必要帯域")):
    s = met[col].dropna()
    if len(s):
        print(f"  {name:<22}: 中央値 {s.median():6.2f} Hz / "
              f"95%tile {s.quantile(0.95):6.2f} Hz / 最大 {s.max():6.2f} Hz")

ok_ratio = 100 * (met["flags"] == "ok").mean()
print(f"\n  品質フラグ ok の割合: {ok_ratio:.1f} %")
if ok_ratio < 100:
    print("  → ok 以外の内訳はセル5の出力、該当ファイル名はCSVで確認できます。")

print("\n💡 カットオフ周波数は、この結果だけでなく")
print("   07_カットオフ周波数決定_全ファイル横断.ipynb で")
print("   両形式をまとめて確認してから決めてください。")
print("   （両対数のPSDグラフ・累積パワー曲線・カットオフ掃引が出せます）")
print("\n🎉 すべての処理が完了しました。")